# 07 — Integración Componente A ↔ Componente B

El **núcleo del proyecto**: acoplar el detector no supervisado (Componente A, un
VAE) con el predictor de rinde (Componente B). Tres análisis que pide la consigna:

1. **Consistencia cruzada** — ¿el score de anomalía del VAE correlaciona con el
   |residuo| del predictor? Si ambos ven la misma estructura, una campaña "rara"
   para el VAE también le cuesta al regresor (Spearman sobre el test).
2. **Aporte del detector al predictor** (lift) — RMSE del predictor **con vs. sin**
   las features del VAE (latente / score), vía `ev.comparar_latente`.
3. **Cuantificación económica** — rinde **contrafactual** bajo clima normal −
   rinde real, × superficie sembrada → pérdida de la sequía 2022/23, comparable al
   benchmark de la BCR (USD 14.140 M).

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))          # componente_b/ (datos, evaluacion)
warnings.filterwarnings('ignore')                  # silenciar ConvergenceWarning de sklearn

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

import datos, evaluacion as ev
from modelos import (LinearRegressor, XGBoostRegressor, NeuralNetRegressor,
                     RandomForestRegressorModel, HistGBMRegressor, StackingRegressorModel)

# Cultivo del estudio (cambiar a 'maiz' para reproducir con maíz).
CULTIVO = 'soja'
ds = datos.prepare(CULTIVO, use_agro=True, enc_smooth=10.0)
print(f'{CULTIVO}: {len(ds.feature_cols)} features | '
      f'train {ds.X_train.shape[0]} filas (≤{datos.TRAIN_END}) | '
      f'test {ds.X_test.shape[0]} filas (≥{datos.TEST_START})')


In [ ]:
import json
_bpath = 'retuning_cv_honesta.json'
BEST_ALL = json.load(open(_bpath, encoding='utf-8'))[CULTIVO] if os.path.exists(_bpath) else {}
def best_of(name, fallback):
    """best-params re-tuneados del modelo `name` (o `fallback` si no hay json)."""
    return BEST_ALL.get(name, {}).get('best_params', fallback)
print('re-tuning disponible:', sorted(k for k in BEST_ALL if not k.startswith('_')) or 'NO (usando fallbacks)')


In [ ]:
import integracion, latente
# Modelo final del predictor: XGBoost re-tuneado.
model = XGBoostRegressor(**best_of('xgb', {}), random_state=42).fit(ds.X_train, ds.y_train)
y_pred = model.predict(ds.X_test)
print('predictor (test):', {k: round(v, 3) for k, v in ev.metricas(ds.y_test, y_pred).items()})
# Features del VAE del Componente A (score + latente), cacheadas.
vf = latente.vae_features(CULTIVO)

## 1. Consistencia cruzada (Spearman)

Correlación de Spearman entre el score de anomalía del VAE y el valor absoluto del
residuo del predictor, sobre el test. Positiva y significativa ⇒ **ambos
componentes detectan la misma estructura** (test de robustez metodológica).

In [ ]:
cc = integracion.consistencia_cruzada(ds, y_pred, vf)
print(f"Spearman(score VAE, |residuo predictor|) = {cc['spearman_rho']:.3f}  "
      f"(p={cc['p_value']:.1e}, n={cc['n']})")
resid = np.abs(ds.y_test - y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(vf['score_test'], resid, s=10, alpha=0.35, color=ev.C_XGB, edgecolors='none')
ax.set_xlabel('score de anomalía del VAE'); ax.set_ylabel('|residuo| del predictor')
ax.set_title(f"Consistencia cruzada  (Spearman ρ={cc['spearman_rho']:.3f})")
plt.tight_layout(); plt.show()

## 2. Aporte del detector al predictor (lift)

RMSE del predictor sobre el dataset base vs. tres estrategias que reusan el VAE:
concatenar su **latente**, usar **solo el latente**, y agregar la categórica
**`es_anomalo`**. Si el detector aporta contexto climático comprimido, debería
bajar el RMSE (sobre todo en campañas extremas).

In [ ]:
tabla_lat = ev.comparar_latente(XGBoostRegressor, best_of('xgb', {}), CULTIVO,
                                fixed={'random_state': 42},
                                vae_kwargs=dict(score_seeds=(42, 43, 44)))
tabla_lat

## 3. Cuantificación económica de la sequía 2022/23

Rinde **contrafactual** (qué habría rendido cada depto con clima normal, poniendo
las features climáticas en su media de train) − rinde real, ponderado por
superficie sembrada. Agregamos la campaña 2022/23 y miramos los deptos más
golpeados.

In [ ]:
y_cf = integracion.contrafactual_normal(model, ds)
res = integracion.resumen_2223(ds, y_cf)
print(f"2022/23 ({res['n_filas']} deptos-cultivo): pérdida total "
      f"{res['perdida_total_tn']:,.0f} tn  ({res['perdida_media_kgha']:.0f} kg/ha promedio)")
res['top_deptos']

In [ ]:
top = res['top_deptos'].head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top['departamento'] + ' (' + top['provincia'].str[:3] + ')',
        top['perdida_tn'] / 1e3, color='#C44E52')
ax.set_xlabel('pérdida estimada (miles de tn)')
ax.set_title('2022/23 — deptos más golpeados (contrafactual clima normal − real)')
plt.tight_layout(); plt.show()

## Conclusión

- **Consistencia cruzada:** la correlación positiva y significativa entre el score
  del VAE y el residuo del predictor indica que ambos componentes captan la misma
  señal de anomalía — el sistema es coherente de punta a punta.
- **Lift del latente:** en línea con el techo estructural del Componente A, el
  latente del VAE aporta poco a la regresión (resume el clima *normalizado por
  depto*, tira la señal espacial/tendencia que domina el rinde). El aporte real del
  acople es **conceptual y económico**, no un salto de RMSE.
- **Cuantificación:** el contrafactual aísla la campaña 2022/23 como pérdida masiva
  concentrada en el sur de Córdoba y Santa Fe, consistente con la sequía histórica y
  el orden de magnitud del benchmark BCR.